# EDA failure PART — OMEXP
Notebook ini membaca schema `analytics` tanpa mengubah tabel sumber. Failure onset adalah `DISMANTLED + CORRECTIVE`; RECON tidak dipakai sebagai waktu operasional. Unit analisis adalah snapshot 30-harian dalam installation cycle yang valid.

In [ ]:
from pathlib import Path
import os, sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
PROJECT_DIR = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_DIR / 'src'))
from database import connect
sns.set_theme(style='whitegrid')
def query(sql, params=None):
    with connect() as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params or ())
            return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])

## 1. Readiness dan validasi anti-leakage

In [ ]:
readiness = query('SELECT * FROM analytics.eda_failure_readiness_summary ORDER BY metric')
display(readiness)
checks = query("""
SELECT
 COUNT(*) - COUNT(DISTINCT (installation_cycle_id, observation_on)) AS duplicate_keys,
 COUNT(*) FILTER (WHERE target_failure_30d AND NOT (next_failure_on > observation_on AND next_failure_on <= observation_on + INTERVAL '30 days')) AS invalid_positive_labels,
 COUNT(*) FILTER (WHERE days_since_installation < 0 OR days_since_last_event < 0 OR days_since_last_failure < 0) AS negative_time_features
FROM analytics.item_observation_30d
""")
display(checks)
assert checks.iloc[0].eq(0).all(), 'Dataset gagal pemeriksaan leakage/key.'

## 2. Target balance dan perubahan antar-era

In [ ]:
yearly = query('SELECT * FROM analytics.eda_failure_rate_by_year ORDER BY observation_year')
display(yearly)
ax = sns.lineplot(data=yearly, x='observation_year', y='positive_percentage', marker='o')
ax.axvline(2025, color='crimson', linestyle='--', label='Detailed repair era')
ax.set(title='Failure dalam 30 hari per tahun', ylabel='Positive (%)', xlabel='Tahun observasi')
ax.legend(); plt.show()

## 3. Kelengkapan fitur

In [ ]:
missing = query('SELECT * FROM analytics.eda_feature_missingness ORDER BY missing_percentage DESC')
display(missing)
sns.barplot(data=missing, y='feature_name', x='missing_percentage').set(title='Missing value fitur', xlabel='Missing (%)', ylabel=''); plt.show()

## 4. Umur cycle sampai failure (hanya timestamp installation valid)

In [ ]:
cycle_stats = query("""SELECT COUNT(*) failure_cycles, ROUND(AVG(days_installed_to_failure)::numeric,2) mean_days, ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY days_installed_to_failure)::numeric,2) median_days, ROUND(PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY days_installed_to_failure)::numeric,2) p90_days FROM analytics.item_installation_cycle WHERE is_initial_model_cohort AND has_observed_failure""")
display(cycle_stats)
durations = query("SELECT LEAST(FLOOR(days_installed_to_failure / 90) * 90, 1800)::int bucket_days, COUNT(*) cycle_count FROM analytics.item_installation_cycle WHERE is_initial_model_cohort AND has_observed_failure GROUP BY 1 ORDER BY 1")
sns.barplot(data=durations, x='bucket_days', y='cycle_count', color='steelblue').set(title='Distribusi umur sampai failure (bucket 90 hari, cap 1800)', xlabel='Hari', ylabel='Cycle'); plt.xticks(rotation=45); plt.show()

## 5. Failure rate menurut model — hanya grup dengan dukungan cukup

In [ ]:
by_model = query("""SELECT item_model_code_clean, COUNT(*) observations, COUNT(DISTINCT item_identifier_clean) items, COUNT(*) FILTER (WHERE target_failure_30d) positives, ROUND(100.0 * COUNT(*) FILTER (WHERE target_failure_30d) / COUNT(*), 3) positive_pct FROM analytics.item_observation_30d WHERE is_training_eligible GROUP BY 1 HAVING COUNT(DISTINCT item_identifier_clean) >= 20 AND COUNT(*) FILTER (WHERE target_failure_30d) >= 10 ORDER BY positives DESC LIMIT 25""")
display(by_model)
sns.barplot(data=by_model, y='item_model_code_clean', x='positive_pct').set(title='Failure rate 30 hari per model (minimum support)', xlabel='Positive (%)', ylabel='Model'); plt.show()

## 6. Distribusi fitur pada positive vs negative
Negative diambil sebagai sampel deterministik agar notebook tidak memuat 1,3 juta baris ke memori.

In [ ]:
sample = query("""SELECT target_failure_30d, days_since_installation, prior_failure_count, prior_corrective_count, prior_relocation_count, prior_distinct_places FROM analytics.item_observation_30d WHERE is_training_eligible AND (target_failure_30d OR MOD(ABS(HASHTEXT(installation_cycle_id || observation_date::text)::bigint), 100) < 3)""")
display(sample.groupby('target_failure_30d').describe().round(2))
plot_data = sample[sample.days_since_installation <= sample.days_since_installation.quantile(.99)].copy()
sns.boxplot(data=plot_data, x='target_failure_30d', y='days_since_installation', showfliers=False).set(title='Umur instalasi vs target 30 hari', xlabel='Failure <= 30 hari', ylabel='Hari sejak installed'); plt.show()

## 7. Split waktu yang direkomendasikan
Gunakan split waktu, bukan random: train sampai 2024, validation 2025, test 2026. Perubahan pencatatan sejak 2025 harus disebut sebagai distribution shift; hasil per-era wajib dilaporkan terpisah. Dataset ini belum melakukan training model.

In [ ]:
splits = query("""SELECT CASE WHEN observation_on < DATE '2025-01-01' THEN 'TRAIN_2013_2024' WHEN observation_on < DATE '2026-01-01' THEN 'VALIDATION_2025' ELSE 'TEST_2026' END split, COUNT(*) observations, COUNT(DISTINCT item_identifier_clean) items, COUNT(*) FILTER (WHERE target_failure_30d) positives FROM analytics.item_observation_30d WHERE is_training_eligible GROUP BY 1 ORDER BY MIN(observation_on)""")
display(splits)